# Gassimulatie – Broncode

Dit notebook bevat de twee centrale simulatiescripts van de masterscriptie
over de sterfspiraal in Vlaamse gasdistributienetwerken.

| Script | Beschrijving |
|---|---|
| `genereer_database.py` | Simuleert de leegloop van het gasnet over 30 jaar en schrijft alle resultaten per jaar en per huishouden weg naar een CSV-database. |
| `bereken_kantelpunt.py` | Berekent voor elke combinatie van parameters de kritische elek/gas-prijsverhouding (kantelpunt) via binair zoeken. |


---
## 1. `genereer_database.py` – Leegloopsimulatie

### Beschrijving

Het script simuleert jaar per jaar welke huishoudens overstappen naar een
warmtepomp op basis van een verdisconteerde terugverdientijdcriterium.
Voor elk huishouden en elk jaar worden alle energiekosten berekend en
weggeschreven naar een CSV-database.

**Invoer:** gebruikersdata (EAN_ID, jaarverbruik, warmtepompprijs, TVT-drempel)
**Uitvoer:** `simulatie_database.csv` met één rij per (jaar × huishouden)


### Importeren en parameters

In [ ]:
import csv   # ingebouwde module om CSV-bestanden te lezen en schrijven
import math  # ingebouwde module voor wiskundige functies

# ── Parameters ────────────────────────────────────────────────────────────────
PARAMETERS = {
    # Invoerbestand
    'csv_gebruikers': 'jaarverbruik_met_wp_prijs (1).xls',

    # Netwerkinvestering
    'investering':   2_500_000,   # EUR totale netwerkinvestering
    'termijn_jaren': 30,          # afschrijvingstermijn in jaren
    'btw':           6,           # % BTW enkel op netkosten
    'alpha':         0.50,        # aandeel vaste netkosten (0=volledig variabel, 1=volledig vast)

    # Warmtepomp
    'cop':           3.5,         # coefficient of performance
    'rendement':     0.9,         # thermisch rendement distributie

    # Overstapcriterium
    'rente':         2,           # % discontovoet voor verdisconteerde TVT
    'acc_tvt':       7,           # globale verschuiving op individuele TVT-drempel
                                  # (buffer = acc_tvt - 7; bij 7 gebruikt elk huishouden
                                  #  exact zijn eigen drempel uit de dataset)

    # Prijstrajecten (None = lineair traject op basis van start/eind)
    'gas_traject':   None,
    'elek_traject':  None,
    'gas_start':     0.075,       # EUR/kWh jaar 0
    'gas_eind':      0.16,        # EUR/kWh jaar 30
    'elek_start':    0.30,        # EUR/kWh jaar 0
    'elek_eind':     0.24,        # EUR/kWh jaar 30
    'convergentie_jaar': 30,      # jaar waarop eindprijs bereikt is

    # True = netkosten bevroren op jaar-0-niveau (geen sterfspiraal op tarief)
    # False = netkosten stijgen naarmate gebruikers vertrekken (sterfspiraal actief)
    'bevries_netkosten': False,

    # Uitvoer
    'uitvoer_csv':  'simulatie_database.csv',
    'jaar_start':    2024,
}

### Hulpfuncties

In [ ]:
def maak_lineair_traject(start, eind, n_jaren=30, convergentie_jaar=30):
    """
    Genereert een lineair prijstraject van 'start' naar 'eind' over
    'convergentie_jaar' stappen. Na het convergentiepunt blijft de prijs
    constant op 'eind'. Geeft een lijst van n_jaren + 1 waarden terug.
    """
    waarden = []
    for t in range(n_jaren + 1):
        # Bepaal hoe ver we in het traject zitten (0 = begin, 1 = einde).
        frac = min(t, convergentie_jaar) / convergentie_jaar
        # Lineaire interpolatie tussen start- en eindprijs.
        waarden.append(start + (eind - start) * frac)
    return waarden


def lees_gebruikers(pad):
    """
    Leest de gebruikersdata uit het CSV-bestand en retourneert een gesorteerde
    lijst van huishoudens op basis van jaarverbruik (laag naar hoog).
    """
    with open(pad, newline='', encoding='utf-8') as f:
        rijen = list(csv.DictReader(f))
    gebruikers = []
    for r in rijen:
        v = float(r['Jaarverbruik_KWh'])
        if v <= 0:
            continue
        gebruikers.append({
            'ean_id':  r['EAN_ID'],
            'vg':      v,
            'wp':      float(r['Warmtepomp_Prijs']),
            'acc_tvt': float(r['Terugverdientijd']),
        })
    gebruikers.sort(key=lambda g: g['vg'])
    return gebruikers


def bereken_tvt(wp_prijs, besparing, dr, max_jaren=30):
    """
    Berekent de verdisconteerde terugverdientijd (TVT).

    De TVT is het kleinste t waarvoor:
        sum_{j=1}^{t} besparing / (1 + r)^j  >=  wp_prijs

    Geeft None terug als de som na max_jaren de investeringskost niet bereikt.
    """
    if besparing <= 0:
        return None  # negatieve besparing: overstap is nooit rendabel
    pv = 0.0
    for t in range(1, max_jaren + 1):
        pv += besparing / (1 + dr) ** t  # verdisconteerde besparing voor jaar t
        if pv >= wp_prijs:
            return t
    return None

### Simulatiefunctie

In [ ]:
def simuleer(p):
    """
    Voert de volledige simulatie uit en retourneert een lijst van dicts,
    één per (jaar, huishouden).
    """
    gebruikers = lees_gebruikers(p['csv_gebruikers'])
    n_jaren    = p['termijn_jaren']
    dr         = p['rente'] / 100
    R          = p['investering'] / n_jaren  # jaarlijkse netwerkkostlast (EUR/jaar)
    buffer     = p['acc_tvt'] - 7            # verschuiving op individuele TVT-drempel

    # Prijstrajecten opbouwen
    gas_traject  = p['gas_traject'] or maak_lineair_traject(
        p['gas_start'], p['gas_eind'], n_jaren, p['convergentie_jaar'])
    elek_traject = p['elek_traject'] or maak_lineair_traject(
        p['elek_start'], p['elek_eind'], n_jaren, p['convergentie_jaar'])

    # Lijst die bijhoudt in welk jaar elk huishouden is overgestapt
    jaar_uitstap = [None] * len(gebruikers)
    records      = []

    # Bevroren nettarieven op jaar-0-niveau
    N0  = len(gebruikers)
    Vg0 = sum(g['vg'] for g in gebruikers)
    F0  = (p['alpha'] * R) / N0
    pv0 = ((1 - p['alpha']) * R) / Vg0

    for t in range(n_jaren + 1):
        jaar_label = p['jaar_start'] + t
        gas_t      = gas_traject[t]
        elek_t     = elek_traject[t]

        actief = [i for i, j in enumerate(jaar_uitstap) if j is None]
        N      = len(actief)
        Vg     = sum(gebruikers[i]['vg'] for i in actief) if N > 0 else 0.0

        # Nettarieven berekenen
        if p['bevries_netkosten']:
            F  = F0   # bevroren: geen sterfspiraal-effect op tarief
            pv = pv0
        else:
            F  = (p['alpha'] * R) / N        if N > 0  else 0.0
            pv = ((1 - p['alpha']) * R) / Vg if Vg > 0 else 0.0

        for i, g in enumerate(gebruikers):
            al_uitgestapt = jaar_uitstap[i] is not None

            # Energiekosten berekenen
            netkost_vast     = F
            netkost_variabel = pv * g['vg']
            netkost_excl     = netkost_vast + netkost_variabel
            netkost_incl     = netkost_excl * (1 + p['btw'] / 100)      # incl. BTW
            gas_factuur      = gas_t * g['vg'] + netkost_incl            # totale gasfactuur
            elek_verbruik    = g['vg'] * p['rendement'] / p['cop']       # verbruik warmtepomp
            elek_factuur     = elek_t * elek_verbruik                    # elektriciteitsfactuur
            besparing        = gas_factuur - elek_factuur                # jaarlijkse besparing
            tvt              = bereken_tvt(g['wp'], besparing, dr)       # verdisconteerde TVT

            if al_uitgestapt or t == 0:
                status = 'uitgestapt' if al_uitgestapt else 'actief'
            else:
                # Overstapcriterium: TVT <= individuele drempel + buffer
                stapt_over = (tvt is not None and
                              tvt <= max(1, g['acc_tvt'] + buffer))
                if stapt_over:
                    jaar_uitstap[i] = jaar_label
                status = 'net_uitgestapt' if stapt_over else 'actief'

            records.append({
                'jaar':                jaar_label,
                'ean_id':              g['ean_id'],
                'jaarverbruik_kwh':    g['vg'],
                'wp_prijs_eur':        g['wp'],
                'acc_tvt_individueel': g['acc_tvt'],
                'gas_prijs':           round(gas_t, 5),
                'elek_prijs':          round(elek_t, 5),
                'n_actief':            N,
                'netkost_vast':        round(netkost_vast, 4),
                'netkost_variabel':    round(netkost_variabel, 4),
                'netkost_incl_btw':    round(netkost_incl, 4),
                'gas_factuur':         round(gas_factuur, 4),
                'elek_verbruik_kwh':   round(elek_verbruik, 4),
                'elek_factuur':        round(elek_factuur, 4),
                'besparing':           round(besparing, 4),
                'tvt_verdisconteerd':  tvt,
                'status':              status,
                'jaar_uitstap':        jaar_uitstap[i],
            })

    return records

### Uitvoer en uitvoeren

In [ ]:
def schrijf_csv(records, pad):
    """Schrijft de lijst van records weg naar een CSV-bestand."""
    if not records:
        print("Geen records om te schrijven.")
        return
    kolomnamen = list(records[0].keys())
    with open(pad, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=kolomnamen)
        writer.writeheader()
        writer.writerows(records)
    print(f"  {len(records)} records weggeschreven naar '{pad}'")


# ── Uitvoeren ─────────────────────────────────────────────────────────────────
records = simuleer(PARAMETERS)
schrijf_csv(records, PARAMETERS['uitvoer_csv'])

# Korte samenvatting
jaren        = sorted(set(r['jaar'] for r in records))
n_start      = next(r['n_actief'] for r in records if r['jaar'] == jaren[0])
actief_eind  = [r for r in records if r['jaar'] == jaren[-1] and r['status'] == 'actief']
n_eind       = len(actief_eind)
gem_nk       = sum(r['netkost_incl_btw'] for r in actief_eind) / n_eind if n_eind > 0 else 0

print(f"  Overblijvers {jaren[0]}:  {n_start}")
print(f"  Overblijvers {jaren[-1]}: {n_eind}  ({n_eind/n_start*100:.1f}%)")
print(f"  Gem. netkost {jaren[-1]}: EUR{gem_nk:.2f}/jaar")

---
## 2. `bereken_kantelpunt.py` – Kantelpuntanalyse

### Beschrijving

Het script berekent voor elke combinatie van parameters de **kritische
elek/gas-prijsverhouding** (het kantelpunt) via binair zoeken.

Het kantelpunt is de prijsverhouding waarbij de simulatie kantelt van een
stabiel netwerk naar een volledige sterfspiraal. Beneden dit punt verlaten
voldoende huishoudens het net om een zelfversterkende leegloop te starten.

De prijzen worden afgeleid uit het geometrisch midden P_ref:

$$\text{gas\_prijs} = \frac{P_{ref}}{\sqrt{\text{ratio}}}, \quad
\text{elek\_prijs} = P_{ref} \cdot \sqrt{\text{ratio}}$$

**Invoer:** gebruikersdata + parametercombinaties
**Uitvoer:** `kantelpunt_database.csv` met één rij per combinatie


### Importeren en parameters

In [ ]:
import csv
import math
import itertools  # voor het Cartesisch product van parametercombinaties

PARAMETERS = {
    # Invoerbestand
    'csv_gebruikers': 'jaarverbruik_met_wp_prijs (1).xls',

    # Vaste netwerkparameters
    'investering':    2_500_000,
    'termijn_jaren':  30,
    'btw':            6,
    'rendement':      0.9,

    # Zoekbereik binair zoeken
    'zoek_lo':  0.20,    # ondergrens prijsverhouding
    'zoek_hi':  6.0,     # bovengrens prijsverhouding
    'zoek_tol': 0.002,   # convergentiedrempel
    'zoek_iter': 40,     # maximaal aantal iteraties

    # Variatiereeksen: elke combinatie wordt doorgerekend
    'acc_tvt_reeks': list(range(5, 16)),          # globale TVT-verschuiving (5-15 jaar)
    'rente_reeks':   list(range(0, 8)),            # discontovoet (0%-7%)
    'cop_reeks':     [3.0, 3.5, 4.0, 4.5, 5.0],  # coefficient of performance
    'alpha_reeks':   [0.0, 0.25, 0.50, 0.75, 1.0],# aandeel vaste netkosten
    'p_ref_reeks':   [0.100, 0.125, 0.150, 0.175, 0.200],  # geometrisch midden (EUR/kWh)

    'uitvoer_csv': 'kantelpunt_database.csv',
}

### Hulpfuncties

In [ ]:
def lees_gebruikers(pad):
    """Leest gebruikersdata in, filtert verbruik <= 0, sorteert op verbruik."""
    with open(pad, newline='', encoding='utf-8') as f:
        rijen = list(csv.DictReader(f))
    gebruikers = []
    for r in rijen:
        v = float(r['Jaarverbruik_KWh'])
        if v <= 0:
            continue
        gebruikers.append({
            'vg':  v,
            'wp':  float(r['Warmtepomp_Prijs']),
            'tvt': float(r['Terugverdientijd']),
        })
    gebruikers.sort(key=lambda g: g['vg'])
    return gebruikers


def bereken_tvt(wp_prijs, besparing, dr, max_jaren=30):
    """
    Verdisconteerde terugverdientijd: kleinste t waarvoor
        sum_{j=1}^{t} besparing / (1+r)^j >= wp_prijs.
    Geeft math.inf als niet bereikt binnen max_jaren.
    """
    if besparing <= 0:
        return math.inf
    pv = 0.0
    for t in range(1, max_jaren + 1):
        pv += besparing / (1 + dr) ** t
        if pv >= wp_prijs:
            return t
    return math.inf


def is_spiral(gebruikers, ratio, acc_tvt, rente, cop, alpha, p_ref, btw, R, rendement):
    """
    Simuleert 30 jaar bij de gegeven prijsverhouding.
    Geeft True als alle huishoudens het gasnet verlaten (volledige sterfspiraal),
    anders False.

    Prijzen via geometrisch midden P_ref:
        gas_prijs  = P_ref / sqrt(ratio)
        elek_prijs = P_ref * sqrt(ratio)
    """
    buffer     = acc_tvt - 7
    gas_prijs  = p_ref / math.sqrt(ratio)
    elek_prijs = p_ref * math.sqrt(ratio)
    dr         = rente / 100
    overstap   = [None] * len(gebruikers)

    for jaar in range(1, 31):
        actief = [i for i, j in enumerate(overstap) if j is None]
        N = len(actief)
        if N == 0:
            return True  # iedereen vertrokken: volledige spiraal

        # Nettarieven stijgen naarmate N en Vg dalen (sterfspiraal-mechanisme)
        Vg  = sum(gebruikers[i]['vg'] for i in actief)
        F   = (alpha * R) / N
        var = ((1 - alpha) * R) / Vg if Vg > 0 else 0.0

        for i in actief:
            g = gebruikers[i]
            v = g['vg']

            nk_incl   = (F + var * v) * (1 + btw / 100)   # netkosten incl. BTW
            gas_f     = gas_prijs * v + nk_incl             # totale gasfactuur
            elek_f    = elek_prijs * (v * rendement / cop)  # elektriciteitsfactuur warmtepomp
            besparing = gas_f - elek_f

            tvt = bereken_tvt(g['wp'], besparing, dr)

            # Overstapcriterium: TVT <= individuele drempel + buffer
            if tvt != math.inf and tvt <= max(1, g['tvt'] + buffer):
                overstap[i] = jaar

    return sum(1 for j in overstap if j is None) == 0

### Binair zoeken naar het kantelpunt

In [ ]:
def vind_kantelpunt(gebruikers, acc_tvt, rente, cop, alpha, p_ref, p):
    """
    Zoekt via binair zoeken naar de kritische prijsverhouding (kantelpunt).

    Het algoritme halveert het interval [zoek_lo, zoek_hi] totdat het
    verschil kleiner is dan zoek_tol. Geeft None terug als geen kantelpunt
    bestaat binnen het zoekbereik.
    """
    R = p['investering'] / p['termijn_jaren']
    kwargs = dict(btw=p['btw'], R=R, rendement=p['rendement'])

    # Grenzen controleren: lage ratio moet spiraal geven, hoge ratio stabiel
    if not is_spiral(gebruikers, p['zoek_lo'], acc_tvt, rente, cop, alpha, p_ref, **kwargs):
        return None  # zelfs laagste ratio geeft geen spiraal
    if is_spiral(gebruikers, p['zoek_hi'], acc_tvt, rente, cop, alpha, p_ref, **kwargs):
        return None  # zelfs hoogste ratio geeft spiraal: buiten bereik

    lo, hi = p['zoek_lo'], p['zoek_hi']
    for _ in range(p['zoek_iter']):
        mid = (lo + hi) / 2
        if is_spiral(gebruikers, mid, acc_tvt, rente, cop, alpha, p_ref, **kwargs):
            lo = mid   # mid geeft spiraal: kantelpunt ligt hoger
        else:
            hi = mid   # mid is stabiel: kantelpunt ligt lager
        if hi - lo < p['zoek_tol']:
            break      # voldoende nauwkeurigheid bereikt

    return (lo + hi) / 2

### Berekening en uitvoer

In [ ]:
def schrijf_csv(records, pad):
    """Schrijft de lijst van records weg naar een CSV-bestand."""
    if not records:
        print("Geen records om te schrijven.")
        return
    kolomnamen = list(records[0].keys())
    with open(pad, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=kolomnamen)
        writer.writeheader()
        writer.writerows(records)
    print(f"  {len(records)} records weggeschreven naar '{pad}'")


# ── Uitvoeren ─────────────────────────────────────────────────────────────────
p = PARAMETERS
gebruikers = lees_gebruikers(p['csv_gebruikers'])
print(f"{len(gebruikers)} gebruikers ingelezen.")

n_totaal = (len(p['acc_tvt_reeks']) * len(p['rente_reeks']) *
            len(p['cop_reeks'])     * len(p['alpha_reeks']) *
            len(p['p_ref_reeks']))
print(f"{n_totaal} combinaties te berekenen...\n")

records = []
teller  = 0

# Doorloop alle combinaties via het Cartesisch product
for acc_tvt, rente, cop, alpha, p_ref in itertools.product(
        p['acc_tvt_reeks'], p['rente_reeks'],
        p['cop_reeks'],     p['alpha_reeks'],
        p['p_ref_reeks']):

    kantelpunt = vind_kantelpunt(gebruikers, acc_tvt, rente, cop, alpha, p_ref, p)

    records.append({
        'acc_tvt':    acc_tvt,
        'rente':      rente,
        'cop':        cop,
        'alpha':      alpha,
        'p_ref':      round(p_ref, 4),
        'kantelpunt': round(kantelpunt, 4) if kantelpunt is not None else None,
        'gas_bij_kp':  round(p_ref / math.sqrt(kantelpunt), 4) if kantelpunt else None,
        'elek_bij_kp': round(p_ref * math.sqrt(kantelpunt), 4) if kantelpunt else None,
    })

    teller += 1
    if teller % 500 == 0:
        print(f"  {teller}/{n_totaal} klaar...")

print()
schrijf_csv(records, p['uitvoer_csv'])
print("\nKlaar.")